# Excel explore — профиль выгрузки Querulus

Локально на закрытом контуре: укажите путь к боевому `.xlsx` в `EXCEL_PATH`.

Тетрадка строит **агрегатный** текстовый отчёт (профиль колонок, денежные сверки, кросстабы флагов) — его можно скопировать в чат. Сырые номера инцидентов в отчёт не попадают.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
for _p in (SRC, PROJECT_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

NOTEBOOK_DIR = PROJECT_ROOT / "monitoring" / "фин. эффекты"
DATA_DIR = NOTEBOOK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT", PROJECT_ROOT)
print("DATA_DIR", DATA_DIR)

In [ ]:
from IPython.display import Markdown, display

from querulus.fin_effect.excel_explore import load_excel, run_explore, save_explore_report
from querulus.fin_effect.excel_monitoring import write_synthetic_claims_excel

# Боевой файл на контуре — подставьте свой путь.
# Пока файла нет: GENERATE_SYNTHETIC=True создаст data/querulus_claims_synthetic.xlsx.
GENERATE_SYNTHETIC = True
EXCEL_PATH = DATA_DIR / "querulus_claims_synthetic.xlsx"
# EXCEL_PATH = DATA_DIR / "claims_prod.xlsx"

SAVE_REPORT_MD = True
REPORT_PATH = DATA_DIR / "excel_explore_report.md"

if GENERATE_SYNTHETIC and not Path(EXCEL_PATH).exists():
    write_synthetic_claims_excel(EXCEL_PATH, n_rows=300)
    print("synthetic written", EXCEL_PATH)

df = load_excel(EXCEL_PATH)
print("shape", df.shape)
print("columns", list(df.columns))

In [ ]:
result = run_explore(df)
display(result.profile)
display(result.reconciliations)
for name, table in result.crosstabs.items():
    print(name)
    display(table)
print("candidates", result.candidates)

## Текстовый отчёт для копипаста

Скопируйте вывод следующей ячейки (или содержимое `data/excel_explore_report.md`) в чат.

In [ ]:
display(Markdown(result.report))
print(result.report)
if SAVE_REPORT_MD:
    save_explore_report(result.report, REPORT_PATH)
    print("saved", REPORT_PATH)